#  Technical Operations Simulation
## Security Company


- **CRM** → `raw_clients.csv` — client accounts, contract types, monthly fees
- **Field Ops system** → `raw_installations.csv` — installation jobs, engineers, systems installed
- **Callout log** → `raw_callouts.csv` — reactive maintenance calls, fault types, SLA response

1. Load and audit each dataset
2. Clean and standardise
3. Consolidate into a master table using SQL
4. Engineer KPIs for the reporting framework
5. Export clean data ready for Power BI

---

In [1]:
import pandas as pd
import numpy as np
import sqlite3

## Audit and clean client data CSV

In [3]:
clients = pd.read_csv("raw_data/raw_clients.csv")

print("=== Client Data ===")
print(clients.shape)
clients.head()

=== Client Data ===
(30, 8)


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,clt001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,NaN
1,CLT002,tesco express hackney,Warehouse,Kent,Dave Okafor,24/11/2022,Monitoring Only,150.0
2,CLT003,Barclays Bank Stratford,Retail,East London,Dave Okafor,12/08/2021,Maintenance,120.0
3,CLT004,Premier Inn Bethnal Green,Transport Hub,Kent,Priya shah,2022-03-06,Maintenance,1000.0
4,CLT005,DHL Warehouse Dagenham,Office,East London,dave okafor,09/03/2022,Full Service,350.0


#### Check for null values and fill them with the median 

In [4]:
print(clients.isnull().sum())

client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    4
dtype: int64


In [5]:
median_fee = clients['monthly_fee_gbp'].median()
clients['monthly_fee_gbp'] = clients['monthly_fee_gbp'].fillna(median_fee)
print(clients.isnull().sum()) 

client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    0
dtype: int64


In [6]:
# Spot IDs that don't match the expected format
clients[clients['client_id'] != clients['client_id'].str.upper()]

,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,clt001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
6,clt007,Lidl Walthamstow,Office,South London,Priya shah,2021-09-28,Monitoring Only,1000.0
12,clt013,Poundland Ilford,Hospitality,West London,Priya Shah,2022-10-18,Monitoring Only,250.0
18,clt019,Argos Barking,Transport Hub,North London,Lee Hutchins,2022-09-03,Monitoring Only,250.0
24,clt025,KFC Stratford,Retail,East London,Priya Shah,2021-11-11,Maintenance,120.0


In [7]:
clients['client_id'] = clients['client_id'].str.upper() #fix all ID names

In [8]:
clients['account_manager'] = clients['account_manager'].str.strip().str.title()

In [9]:
dupes = clients[clients['client_name'].str.lower().duplicated(keep=False)]
dupes

,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
1,CLT002,tesco express hackney,Warehouse,Kent,Dave Okafor,24/11/2022,Monitoring Only,150.0


### Handling Duplicate: Tesco Express Hackney

Two entries found for the same client (CLT001 and CLT002). 
Confirmed with account manager that CLT001 is the correct record.
CLT002 dropped — incorrect region (Kent vs East London) and lower contract value.

In [10]:
clients = clients.drop(index=1)

In [11]:
print(clients.shape)
clients[clients['client_name'].str.lower().str.contains('tesco')]

(29, 8)


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0


In [12]:
print(clients['contract_start'].tolist())

['2021-10-09', '12/08/2021', '2022-03-06', '09/03/2022', '05/04/2021', '2021-09-28', '28/10/2021', '09/11/2022', '2022-01-25', '15/01/2022', '16/09/2022', '2022-10-18', '02/02/2021', '19/11/2021', '2021-09-29', '21/08/2022', '04/04/2021', '2022-09-03', '26/06/2022', '19/10/2022', '2021-01-04', '17/10/2022', '28/06/2022', '2021-11-11', '13/05/2022', '19/06/2021', '2021-11-16', '04/05/2021', '22/07/2022']


### Fix: Mixed Date Formats in contract_start

Audit revealed two date formats present: DD/MM/YYYY and YYYY-MM-DD.
Wrote a custom parser to handle both and standardised to datetime64.

In [13]:
def parse_date(d):
    for fmt in ('%d/%m/%Y', '%Y-%m-%d'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            continue
    return pd.NaT

clients['contract_start'] = clients['contract_start'].apply(parse_date)

In [14]:
print(clients['contract_start'].head(10).tolist())
print(clients['contract_start'].dtype)

[Timestamp('2021-10-09 00:00:00'), Timestamp('2021-08-12 00:00:00'), Timestamp('2022-03-06 00:00:00'), Timestamp('2022-03-09 00:00:00'), Timestamp('2021-04-05 00:00:00'), Timestamp('2021-09-28 00:00:00'), Timestamp('2021-10-28 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-01-25 00:00:00'), Timestamp('2022-01-15 00:00:00')]
datetime64[us]


In [15]:
print(clients.shape)
print(clients.isnull().sum())
print(clients.dtypes)
clients.head()

(29, 8)
client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    0
dtype: int64
client_id                     str
client_name                   str
site_type                     str
region                        str
account_manager               str
contract_start     datetime64[us]
contract_type                 str
monthly_fee_gbp           float64
dtype: object


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
2,CLT003,Barclays Bank Stratford,Retail,East London,Dave Okafor,2021-08-12,Maintenance,120.0
3,CLT004,Premier Inn Bethnal Green,Transport Hub,Kent,Priya Shah,2022-03-06,Maintenance,1000.0
4,CLT005,DHL Warehouse Dagenham,Office,East London,Dave Okafor,2022-03-09,Full Service,350.0
5,CLT006,NHS Whipps Cross,Warehouse,South London,Dave Okafor,2021-04-05,Full Service,150.0


## Audit and clean installation data CSV

In [38]:
installs = pd.read_csv("raw_data/raw_installations.csv")
print(installs.shape)
installs.head(5)

(80, 10)


,job_id,client_ref,system_type,engineer,job_date,duration_hrs,install_cost_gbp,status,cameras_installed,warranty_months
0,JOB0001,CLT003,Intruder Alarm,Mike Tanner,2023-01-31,7.4,3000.0,Completed,NaN,24.0
1,JOB0002,CLT009,Intruder Alarm,Connor Mills,2023-09-06,2.4,NaN,Completed,13.0,24.0
2,JOB0003,CLT004,Intercom,mike tanner,2023-02-19,3.8,4500.0,In Progress,NaN,12.0
3,JOB0004,CLT022,CCTV,mike tanner,2023-02-20,6.2,1200.0,Completed,NaN,24.0
4,JOB0005,CLT018,Intruder Alarm,mike tanner,2023-08-18,2.7,NaN,Completed,14.0,NaN


In [39]:
print(installs.isnull().sum())

job_id                0
client_ref            0
system_type           0
engineer              0
job_date              0
duration_hrs          0
install_cost_gbp      7
status                0
cameras_installed    53
warranty_months      14
dtype: int64


In [40]:
median_install_cost = installs['install_cost_gbp'].median()

installs['install_cost_gbp'] = installs['install_cost_gbp'].fillna(median_install_cost)

In [41]:
print(installs['engineer'].value_counts())

engineer
Raj Patel       17
Mike Tanner     16
Connor Mills    14
mike tanner     14
Yemi Adeyemi    10
Sharon Webb      9
Name: count, dtype: int64


In [42]:
installs['engineer'] = installs['engineer'].str.strip().str.title()

In [43]:
print(installs['engineer'].value_counts())

engineer
Mike Tanner     30
Raj Patel       17
Connor Mills    14
Yemi Adeyemi    10
Sharon Webb      9
Name: count, dtype: int64


In [45]:
installs['job_date'] = installs['job_date'].apply(parse_date)

In [47]:
print(installs['status'].value_counts())

status
Completed           43
Failed - Revisit    13
In Progress         12
Pending             12
Name: count, dtype: int64


In [48]:
print(installs['cameras_installed'].value_counts(dropna=False))

cameras_installed
NaN     53
13.0     5
8.0      4
15.0     3
11.0     2
2.0      2
1.0      2
14.0     1
0.0      1
9.0      1
4.0      1
10.0     1
16.0     1
5.0      1
3.0      1
7.0      1
Name: count, dtype: int64


In [49]:
installs[installs['cameras_installed'].isna()]['system_type'].value_counts()

system_type
CCTV + Alarm      11
Intercom          10
Fire Alarm        10
Intruder Alarm     9
CCTV               9
Access Control     4
Name: count, dtype: int64

In [50]:
installs['cameras_installed'] = installs['cameras_installed'].fillna(0).astype(int)

In [51]:
print(installs['warranty_months'].value_counts(dropna=False))

warranty_months
24.0    35
36.0    19
NaN     14
12.0    12
Name: count, dtype: int64


In [53]:
waranty_median = installs['warranty_months'].median()
installs['warranty_months'] = installs['warranty_months'].fillna(waranty_median)

In [55]:
print(installs.shape)
print(installs.isnull().sum())
print(installs.dtypes)
installs.head()

(80, 10)
job_id               0
client_ref           0
system_type          0
engineer             0
job_date             0
duration_hrs         0
install_cost_gbp     0
status               0
cameras_installed    0
warranty_months      0
dtype: int64
job_id                          str
client_ref                      str
system_type                     str
engineer                        str
job_date             datetime64[us]
duration_hrs                float64
install_cost_gbp            float64
status                          str
cameras_installed             int64
warranty_months             float64
dtype: object


,job_id,client_ref,system_type,engineer,job_date,duration_hrs,install_cost_gbp,status,cameras_installed,warranty_months
0,JOB0001,CLT003,Intruder Alarm,Mike Tanner,2023-01-31,7.4,3000.0,Completed,0,24.0
1,JOB0002,CLT009,Intruder Alarm,Connor Mills,2023-09-06,2.4,2000.0,Completed,13,24.0
2,JOB0003,CLT004,Intercom,Mike Tanner,2023-02-19,3.8,4500.0,In Progress,0,12.0
3,JOB0004,CLT022,CCTV,Mike Tanner,2023-02-20,6.2,1200.0,Completed,0,24.0
4,JOB0005,CLT018,Intruder Alarm,Mike Tanner,2023-08-18,2.7,2000.0,Completed,14,24.0
